# Eksploracja danych — brakujące wartości

**Cel zajęć:** nauczyć się *znajdować* braki (także te ukryte), *diagnozować*, czy są przypadkowe, i *świadomie wybierać*, czy je usunąć, czy uzupełnić.

**Plan:**
1. Rozgrzewka na zbiorze **Titanic** — te same kroki co na wykładzie.
2. Zadanie główne: **oferty mieszkań** (`nieruchomosci.csv`) — prawdziwy bałagan.

**Zadania:** 🟢 podstawowe (obowiązkowe), 🔵 dla ambitnych.

**Jak pracować:** w Colabie wybierz *Plik → Zapisz kopię na Dysku*, wgraj plik `nieruchomosci.csv` (ikona folderu po lewej → *Prześlij*), rozwiąż zadania w miejscach `# TODO`, a gotowy notatnik umieść w swoim repozytorium na GitHubie.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({"figure.figsize": (7, 4), "font.size": 11})

URL_TITANIC = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
titanic = pd.read_csv("titanic.csv") if Path("titanic.csv").exists() else pd.read_csv(URL_TITANIC)

if not Path("nieruchomosci.csv").exists():
    print("⚠️ Wgraj plik nieruchomosci.csv do bieżącego katalogu (w Colabie: ikona folderu → Prześlij).")

print(titanic.shape)
titanic.head()

---
# Część 1. Rozgrzewka: Titanic

| kolumna | znaczenie |
|---|---|
| `Survived` | czy pasażer przeżył (1 = tak) |
| `Pclass` | klasa biletu (1, 2, 3) |
| `Sex`, `Age` | płeć, wiek |
| `SibSp`, `Parch` | liczba rodzeństwa/małżonków oraz rodziców/dzieci na pokładzie |
| `Fare` | cena biletu (£) |
| `Cabin` | numer kabiny |
| `Embarked` | port zaokrętowania: C — Cherbourg, Q — Queenstown, S — Southampton |

### 🟢 Zadanie 1: ile jest braków?

1. Policz liczbę i odsetek braków w każdej kolumnie. Pokaż tylko kolumny, w których braki występują, posortowane malejąco.
2. Narysuj poziomy wykres słupkowy odsetka braków (`.plot.barh()`), z opisem osi.

In [ ]:
braki = pd.DataFrame({
    "liczba": titanic.isna().sum(),
    "odsetek [%]": titanic.isna().mean() * 100,
})
braki = braki[braki["liczba"] > 0].sort_values("liczba", ascending=False)
display(braki.round(1))
print("łączna liczba braków:", titanic.isna().sum().sum())

fig, ax = plt.subplots(figsize=(7, 3))
braki["odsetek [%]"].sort_values().plot.barh(ax=ax, color="steelblue")
ax.bar_label(ax.containers[0], fmt="%.1f%%", padding=3)
ax.set_xlim(0, 90)
ax.set_xlabel("Odsetek braków [%]")
ax.set_ylabel("Kolumna")
ax.set_title("Braki danych w zbiorze Titanic")
plt.tight_layout()
plt.show()

### 🟢 Zadanie 2: mapa braków

Narysuj mapę braków (`sns.heatmap(titanic.isna(), cbar=False)`) dla kolumn `Age`, `Cabin`, `Embarked` — najpierw z wierszami w oryginalnej kolejności, a potem **posortowanymi według klasy** (`Pclass`). Dwa wykresy obok siebie.

Co widać na drugim wykresie, czego nie było widać na pierwszym?

In [ ]:
kolumny = ["Age", "Cabin", "Embarked"]
from matplotlib.colors import ListedColormap
cmap_braki = ListedColormap(["#dbe6ee", "#d95f02"])

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

sns.heatmap(titanic[kolumny].isna(), cbar=False, yticklabels=False, cmap=cmap_braki, ax=axes[0])
axes[0].set_title("Oryginalna kolejność wierszy")
axes[0].set_ylabel("Pasażerowie (891)")

kolejnosc = titanic.sort_values("Pclass", kind="stable").index
sns.heatmap(titanic.loc[kolejnosc, kolumny].isna(), cbar=False, yticklabels=False, cmap=cmap_braki, ax=axes[1])
axes[1].set_title("Posortowane wg klasy (1. → 3.)")
axes[1].set_ylabel("")

fig.suptitle("Mapa braków (pomarańczowy = brak danych)")
plt.tight_layout()
plt.show()

print("odsetek braków [%] w każdej klasie:")
print(titanic[kolumny].isna().groupby(titanic["Pclass"]).mean().mul(100).round(1))

*Twoja odpowiedź:* …

Po posortowaniu według klasy widać, że braki nie są losowe. Cabin najczęściej brakuje w 2. i 3. klasie, Age najczęściej w 3. klasie, a braki w Embarked są pojedyncze i występują w 1. klasie. Oznacza to, że braki mogą zależeć od klasy biletu, dlatego warto analizować je po posortowaniu danych według tej zmiennej.

### 🟢 Zadanie 3: czy braki wieku są losowe?

1. Utwórz serię `brak_wieku = titanic["Age"].isna()`.
2. Policz odsetek braków wieku w każdej klasie i osobno dla każdej płci.
3. Porównaj odsetek ocalałych wśród osób ze znanym wiekiem i bez wieku.
4. Narysuj wyniki z punktów 2 i 3 na dwóch wykresach słupkowych.
5. Czy braki wieku to MCAR? Co by się stało z odsetkiem ocalałych, gdybyśmy usunęli wiersze bez wieku?

In [ ]:
brak_wieku = titanic["Age"].isna()

po_klasie = brak_wieku.groupby(titanic["Pclass"]).mean() * 100
po_plci = brak_wieku.groupby(titanic["Sex"]).mean() * 100
print("braki wieku wg klasy [%]:\n", po_klasie.round(1), "\n")
print("braki wieku wg płci [%]:\n", po_plci.round(1), "\n")
print(f"3. klasa vs 2. klasa: braki częstsze {po_klasie[3] / po_klasie[2]:.1f} razy\n")

ocaleni = titanic.groupby(brak_wieku)["Survived"].mean() * 100
ocaleni.index = ["wiek znany", "brak wieku"]
print("odsetek ocalałych [%]:\n", ocaleni.round(1), "\n")
print(f"cały zbiór: {titanic['Survived'].mean() * 100:.1f}% | po dropna(Age): {titanic.dropna(subset=['Age'])['Survived'].mean() * 100:.1f}%")


fig, axes = plt.subplots(1, 2, figsize=(12, 4))

etykiety = ["1. klasa", "2. klasa", "3. klasa", "kobiety", "mężczyźni"]
wartosci = list(po_klasie.values) + [po_plci["female"], po_plci["male"]]
slupki = axes[0].bar(etykiety, wartosci, color=["#1b9e77"] * 3 + ["#7570b3"] * 2)
axes[0].bar_label(slupki, fmt="%.1f%%", padding=2)
axes[0].set_title("Odsetek braków wieku")
axes[0].set_ylabel("Odsetek braków [%]")
axes[0].set_ylim(0, 40)
axes[0].tick_params(axis="x", rotation=20)

slupki = axes[1].bar(ocaleni.index, ocaleni.values, color=["steelblue", "#d95f02"])
axes[1].bar_label(slupki, fmt="%.1f%%", padding=2)
axes[1].set_title("Odsetek ocalałych")
axes[1].set_ylabel("Ocalali [%]")
axes[1].set_ylim(0, 60)

plt.tight_layout()
plt.show()

*Twoja odpowiedź:*

Braki wieku nie są MCAR, ponieważ zależą od innych zmiennych. W 3. klasie brakuje wieku u 27,7% pasażerów, a w 2. klasie tylko u 6,0%. Osoby bez podanego wieku rzadziej przeżyły (29,4% wobec 40,6%). Wskazuje to na mechanizm MAR. Usunięcie 177 wierszy z brakami zawyżyłoby odsetek ocalałych z 38,4% do 40,6%, dlatego mogłoby zniekształcić wyniki analizy.




### 🟢 Zadanie 4: ile kosztuje usuwanie?

Dla czterech wariantów — bez zmian, `dropna(subset=["Embarked"])`, `dropna(subset=["Age"])`, `dropna()` — zbuduj tabelę z kolumnami:
- liczba wierszy,
- odsetek ocalałych,
- odsetek pasażerów 1. klasy.

Który wariant najbardziej zniekształca dane i dlaczego?

In [ ]:
warianty = {
    "bez zmian": titanic,
    "dropna(Embarked)": titanic.dropna(subset=["Embarked"]),
    "dropna(Age)": titanic.dropna(subset=["Age"]),
    "dropna()": titanic.dropna(),
}

tabela = pd.DataFrame({
    nazwa: {
        "liczba wierszy": len(d),
        "ocalali [%]": d["Survived"].mean() * 100,
        "1. klasa [%]": (d["Pclass"] == 1).mean() * 100,
    }
    for nazwa, d in warianty.items()
}).T
tabela["liczba wierszy"] = tabela["liczba wierszy"].astype(int)
tabela.round(1)

*Twoja odpowiedź:* …

Najbardziej zniekształca dane dropna(), ponieważ zostawia tylko 183 z 891 pasażerów, z czego 67,2% przeżyło, a 86,3% podróżowało 1. klasą. Braki w Cabin dotyczą głównie pasażerów 2. i 3. klasy, więc po ich usunięciu próba przestaje być reprezentatywna. dropna(Age) zniekształca dane mniej, a dropna(Embarked) prawie wcale, ponieważ usuwa tylko 2 wiersze.




### 🟢 Zadanie 5: uzupełnianie wieku

1. **Najpierw** dodaj kolumnę-flagę `Age_brak`.
2. Utwórz dwie wersje kolumny wieku:
   - `wiek_srednia` — braki uzupełnione średnią,
   - `wiek_grupy` — braki uzupełnione medianą w grupach (klasa × płeć); wskazówka: `groupby(...)["Age"].transform("median")`.
3. Narysuj trzy histogramy obok siebie (te same przedziały, `bins=np.arange(0, 82, 2)`): znane wartości, `wiek_srednia`, `wiek_grupy`.
4. Porównaj odchylenia standardowe. Która metoda bardziej zniekształca rozkład?

In [ ]:
titanic["Age_brak"] = titanic["Age"].isna()

titanic["wiek_srednia"] = titanic["Age"].fillna(titanic["Age"].mean())
mediana_grupy = titanic.groupby(["Pclass", "Sex"])["Age"].transform("median")
titanic["wiek_grupy"] = titanic["Age"].fillna(mediana_grupy)

print("mediany wieku w grupach (klasa × płeć):")
print(titanic.groupby(["Pclass", "Sex"])["Age"].median().unstack())

bins = np.arange(0, 82, 2)
wersje = [
    ("znane wartości", titanic["Age"].dropna()),
    ("uzupełnione średnią", titanic["wiek_srednia"]),
    ("mediana w grupach (klasa × płeć)", titanic["wiek_grupy"]),
]
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (tytul, dane) in zip(axes, wersje):
    ax.hist(dane, bins=bins, edgecolor="white", color="steelblue")
    ax.set_title(f"{tytul}\n(n = {len(dane)})")
    ax.set_xlabel("Wiek [lata]")
axes[0].set_ylabel("Liczba pasażerów")
plt.tight_layout()
plt.show()

podsumowanie = pd.DataFrame({
    "odch. std.": [d.std() for _, d in wersje],
    "korelacja z Pclass": [d.corr(titanic.loc[d.index, "Pclass"]) for _, d in wersje],
}, index=[t for t, _ in wersje])
podsumowanie.round(2)

*Twoja odpowiedź:* …

Uzupełnianie średnią i medianą zaniża zmienność wieku. Średnia bardziej zniekształca rozkład, ponieważ 177 pasażerów otrzymuje tę samą wartość 29,7 roku. Mediana w grupach lepiej uwzględnia klasę i płeć, ale również tworzy sztuczne skupiska i zmienia korelację wieku z klasą. Dlatego warto zachować flagę Age_brak i porównywać rozkłady przed i po uzupełnieniu.

### 🔵 Zadanie 6 (dla ambitnych): imputacja modelem

Uzupełnij wiek za pomocą `IterativeImputer` z biblioteki scikit-learn, wykorzystując kolumny `Pclass`, `SibSp`, `Parch`, `Fare` oraz płeć zakodowaną jako 0/1. Porównaj histogram i odchylenie standardowe z wynikami zadania 5.

```python
from sklearn.experimental import enable_iterative_imputer  # wymagane!
from sklearn.impute import IterativeImputer
```

In [ ]:
# TODO

---
# Część 2. Zadanie główne: oferty mieszkań

**Dane:** plik `nieruchomosci.csv` — 45 ofert sprzedaży mieszkań w pięciu miastach.

| kolumna | znaczenie |
|---|---|
| `powierzchnia` | powierzchnia mieszkania [m²] |
| `rok_budowy` | rok budowy budynku |
| `pietro`, `liczba_pieter` | piętro mieszkania (0 = parter) i liczba pięter budynku |
| `parking`, `balkon` | Tak / Nie |
| `stan` | stan mieszkania |
| `cena_m2`, `cena_total` | cena za m² i cena całkowita [zł] |

Plik przygotował ktoś, kto różne braki zapisywał na różne sposoby. Twoim zadaniem jest je znaleźć i rozsądnie obsłużyć.

### 🟢 Zadanie 7: pierwszy rzut oka

1. Wczytaj plik bez żadnych dodatkowych parametrów i sprawdź typy kolumn (`dtypes`).
2. Policz braki (`isna().sum()`). Ile braków widzi pandas?
3. Dlaczego kolumna `powierzchnia` nie jest liczbowa, choć zawiera powierzchnię w m²?

In [ ]:
mieszkania = pd.read_csv("nieruchomosci.csv")

print(mieszkania.dtypes)

braki_domyslne = mieszkania.isna().sum()
print("\nbraki widziane przez pandas:")
print(braki_domyslne[braki_domyslne > 0])
print("łącznie:", braki_domyslne.sum())

napisy = mieszkania["powierzchnia"][pd.to_numeric(mieszkania["powierzchnia"], errors="coerce").isna()
                                    & mieszkania["powierzchnia"].notna()]
print("\nnapisy w kolumnie powierzchnia, które nie są liczbą:", napisy.unique().tolist())
mieszkania.head()

*Twoja odpowiedź:* …

Pandas rozpoznaje tylko 8 braków: 4 w powierzchnia i 4 w rok_budowy, ponieważ  N/A i  NA są domyślnie uznawane za braki. Wartości takie jak  brak,  ?, -999, -1 i 0 traktuje jako zwykłe dane. Napisy  brak powodują, że kolumny powierzchnia, rok_budowy i pietro mają typ tekstowy, co utrudnia obliczanie średniej i rysowanie histogramów.

### 🟢 Zadanie 8: „braki  w przebraniu”

1. Dla każdej kolumny tekstowej i każdej kolumny liczbowej obejrzyj występujące wartości (`value_counts(dropna=False)` albo `unique()`). Wypisz wszystkie wartości, które oznaczają brak.
2. **Pułapka:** w kolumnach `powierzchnia` i `pietro` występuje `0`. Czy w obu przypadkach to brak danych?
3. Wczytaj plik ponownie z parametrem `na_values` podanym jako **słownik** (kolumna → lista wartości), tak aby każda kolumna miała tylko  „swoje” oznaczenia braków.
4. Sprawdź typy kolumn i policz braki jeszcze raz.

In [ ]:
surowe = pd.read_csv("nieruchomosci.csv", dtype=str, keep_default_na=False)

print("=== kolumny kategoryczne ===")
for kol in ["miasto", "parking", "balkon", "stan"]:
    print(f"\n--- {kol}")
    print(surowe[kol].value_counts(dropna=False))

print("\n=== kolumny liczbowe: napisy nieliczbowe, wartości specjalne ===")
for kol in ["powierzchnia", "liczba_pokoi", "rok_budowy", "pietro", "liczba_pieter", "cena_m2", "cena_total"]:
    liczby = pd.to_numeric(surowe[kol], errors="coerce")
    nieliczbowe = sorted(surowe.loc[liczby.isna(), kol].unique())
    print(f"{kol:14s} napisy: {nieliczbowe!s:18s} | min = {liczby.min():g} | max = {liczby.max():g} "
          f"| zer: {(liczby == 0).sum()} | ujemnych: {(liczby < 0).sum()}")

*Twoja odpowiedź:* …
W danych znajduje się łącznie 29 braków, podczas gdy Pandas domyślnie rozpoznał tylko 8. Różne kolumny mają własne oznaczenia brakujących wartości, np.  brak,  N/A,  NA,  ? czy wartości liczbowe -999 i -1.

Trzeba uważać na zera: w kolumnie powierzchnia oznaczają brak danych, ale w pietro oznaczają parter, czyli prawidłową wartość. Dlatego oznaczenia braków należy ustawiać osobno dla każdej kolumny, używając słownika na_values i keep_default_na=False.

In [ ]:
braki_slownik = {
    "powierzchnia": ["brak", "N/A", "0"],
    "rok_budowy":   ["NA", "brak"],
    "pietro":       ["brak", "-999"],
    "parking":      ["?"],
    "balkon":       ["?"],
    "cena_m2":      ["-1", "-999"],
}

mieszkania = pd.read_csv("nieruchomosci.csv", na_values=braki_slownik, keep_default_na=False)

print(mieszkania.dtypes, "\n")
braki_po = mieszkania.isna().sum()
print("braki po poprawnym wczytaniu:")
print(braki_po[braki_po > 0])
print("łącznie:", braki_po.sum(), "(wcześniej pandas widział:", braki_domyslne.sum(), ")")

### 🟢 Zadanie 9: dziwna kategoria

Obejrzyj wartości w kolumnie `stan`. Jedna z nich nie pasuje do pozostałych. Co mogła oznaczać i jak ją poprawić? Zaproponuj i zastosuj rozwiązanie (uzasadnij je).

In [ ]:
print(mieszkania["stan"].value_counts())
nietypowa = ~mieszkania["stan"].isin(["Bardzo dobry", "Dobry", "Do remontu"])
display(mieszkania.loc[nietypowa, ["id", "miasto", "dzielnica", "powierzchnia", "rok_budowy", "stan", "cena_m2"]])

miasto_wiersza = mieszkania.loc[nietypowa, "miasto"].iloc[0]
print(f"\nceny za m² w mieście {miasto_wiersza} wg stanu (bez nietypowego wiersza):")
display(mieszkania[(mieszkania["miasto"] == miasto_wiersza) & ~nietypowa]
        .groupby("stan")["cena_m2"].agg(["count", "median", "min", "max"]))


mieszkania["stan"] = mieszkania["stan"].replace({"很好": "Bardzo dobry"})
print("\npo poprawce:")
print(mieszkania["stan"].value_counts())

Wartość 很好 oznacza po chińsku  bardzo dobry, dlatego zastępuję ją tą kategorią zamiast NaN. Potwierdzają to również ceny ofert – 10 200 zł/m² mieści się w przedziale cenowym mieszkań o bardzo dobrym stanie. Ze względu na małą liczbę ofert (2) nie jest to jednak jednoznaczny dowód. W razie wątpliwości można byłoby oznaczyć tę wartość jako NaN i dodać osobną flagę. Po poprawce liczba ofert w kategorii  Bardzo dobry wzrasta z 11 do 12.

### 🟢 Zadanie 10: mapa braków

1. Narysuj odsetek braków w kolumnach (wykres słupkowy) i mapę braków.
2. Czy braki w różnych kolumnach występują w tych samych wierszach? (Wskazówka: `mieszkania.isna().corr()` dla kolumn z brakami.)

In [ ]:
kolumny_braki = [k for k in mieszkania.columns if mieszkania[k].isna().any()]
odsetek = mieszkania[kolumny_braki].isna().mean().sort_values() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [1, 1.2]})

slupki = axes[0].barh(odsetek.index, odsetek.values, color="steelblue")
axes[0].bar_label(slupki, fmt="%.1f%%", padding=3)
axes[0].set_xlim(0, 25)
axes[0].set_xlabel("Odsetek braków [%]")
axes[0].set_ylabel("Kolumna")
axes[0].set_title("Odsetek braków w kolumnach")

from matplotlib.colors import ListedColormap
sns.heatmap(mieszkania[kolumny_braki].isna(), cbar=False, cmap=ListedColormap(["#dbe6ee", "#d95f02"]),
            yticklabels=mieszkania["id"], linewidths=0.3, linecolor="white", ax=axes[1])
axes[1].tick_params(axis="y", labelsize=7)
axes[1].set_ylabel("id oferty")
axes[1].set_title("Mapa braków (pomarańczowy = brak)")
plt.tight_layout()
plt.show()


print("korelacja wskaźników braków (isna) między kolumnami:")
display(mieszkania[kolumny_braki].isna().corr().round(2))

liczba_brakow = mieszkania[kolumny_braki].isna().sum(axis=1)
print("liczba braków w wierszu (liczba ofert):")
print(liczba_brakow.value_counts().sort_index().to_string())
print("\noferty z co najmniej 2 brakami:")
display(mieszkania.loc[liczba_brakow >= 2, ["id", "miasto"] + kolumny_braki])

print("odsetek braków wg miasta [%]:")
display((mieszkania[kolumny_braki].isna().groupby(mieszkania["miasto"]).mean() * 100).round(0))

*Twoja odpowiedź:* …
Braki występują głównie w różnych wierszach. Korelacje wskaźników braków wynoszą od −0,21 do 0,13, więc nie widać wyraźnej zależności między kolumnami. Spośród 45 ofert 19 nie ma braków, 23 mają jeden brak, a tylko 3 mają po dwa (id 15, 27 i 32). Najwięcej braków występuje w kolumnach powierzchnia (20%) i rok_budowy (15,6%). Braki pojawiają się we wszystkich miastach, bez widocznego wzoru. Sugeruje to mechanizm MCAR, ale ze względu na małą liczbę ofert nie można tego jednoznacznie potwierdzić. W praktyce braki można więc uzupełniać osobno dla każdej kolumny.

### 🟢 Zadanie 11: odtwórz, zamiast zgadywać

Sprawdź, czy dla wierszy bez braków zachodzi zależność `cena_total = powierzchnia × cena_m2`. Jeśli tak — wykorzystaj ją, żeby **odtworzyć** brakujące wartości w kolumnach `powierzchnia` i `cena_m2`. Ile braków udało się w ten sposób usunąć?

To jest  ręczne „uzupełnianie” z wykładu: znamy prawdziwą wartość, więc nie musimy jej szacować.

In [ ]:
kompletne = mieszkania.dropna(subset=["powierzchnia", "cena_m2"])
roznica = kompletne["cena_total"] - kompletne["powierzchnia"] * kompletne["cena_m2"]
print(f"wierszy kompletnych: {len(kompletne)}, największa |różnica|: {roznica.abs().max():.2f} zł")

braki_przed = mieszkania[["powierzchnia", "cena_m2"]].isna().sum()
brak_pow = mieszkania["powierzchnia"].isna()
brak_cena = mieszkania["cena_m2"].isna()

mieszkania["powierzchnia"] = mieszkania["powierzchnia"].fillna(mieszkania["cena_total"] / mieszkania["cena_m2"])
mieszkania["cena_m2"] = mieszkania["cena_m2"].fillna(mieszkania["cena_total"] / mieszkania["powierzchnia"])

braki_po11 = mieszkania[["powierzchnia", "cena_m2"]].isna().sum()
print(pd.DataFrame({"przed": braki_przed, "po": braki_po11, "odtworzono": braki_przed - braki_po11}))

print("\nodtworzone powierzchnie [m²]:")
display(mieszkania.loc[brak_pow, ["id", "miasto", "powierzchnia", "cena_m2", "cena_total"]])
print("odtworzone ceny za m² [zł]:")
display(mieszkania.loc[brak_cena, ["id", "miasto", "powierzchnia", "cena_m2", "cena_total"]])

### 🟢 Zadanie 12: pozostałe braki

Dla każdej z kolumn `rok_budowy`, `pietro`, `parking`, `balkon`:
1. **Najpierw** dodaj kolumnę-flagę braku (np. `rok_budowy_brak`).
2. Zdecyduj, co zrobić z brakami, i zaimplementuj decyzję. Do rozważenia:
   - `rok_budowy` — mediana całościowa czy mediana w grupach? Jakie grupy mają sens: `miasto` czy `stan`? Policz mediany w obu podziałach, zanim wybierzesz.
   - `pietro` — są tylko 3 braki; mediana, a może coś mądrzejszego (np. środkowe piętro budynku: `liczba_pieter // 2`)?
   - `parking`, `balkon` — dominanta czy osobna kategoria `"nieznane"`? Co się stanie z odsetkiem mieszkań z parkingiem w każdym z tych wariantów?
3. Sprawdź, że w zbiorze nie ma już braków.

In [ ]:
rok_przed_uzupelnieniem = mieszkania["rok_budowy"].copy()

for kol in ["rok_budowy", "pietro", "parking", "balkon"]:
    mieszkania[f"{kol}_brak"] = mieszkania[kol].isna()
print("flagi (liczba braków):", {k: int(mieszkania[f"{k}_brak"].sum()) for k in ["rok_budowy", "pietro", "parking", "balkon"]})

r = mieszkania["rok_budowy"]
znane = r.notna()
print("\nmediana całościowa roku budowy:", r.median())
print("\nmediany wg miasta:", mieszkania.groupby("miasto")["rok_budowy"].median().to_dict())
print("mediany wg stanu: ", mieszkania.groupby("stan")["rok_budowy"].median().to_dict())

med_calosc = pd.Series(r.median(), index=r.index)
med_miasto = mieszkania.groupby("miasto")["rok_budowy"].transform("median")
med_stan = mieszkania.groupby("stan")["rok_budowy"].transform("median")
for nazwa, med in [("całościowa", med_calosc), ("wg miasta", med_miasto), ("wg stanu", med_stan)]:
    print(f"  średni błąd |rok - mediana {nazwa}|: {(r[znane] - med[znane]).abs().mean():.2f} lat")

mieszkania["rok_budowy"] = r.fillna(med_stan).round().astype(int)

p = mieszkania["pietro"]
zn = p.notna()
print(f"\npiętro - korelacja z liczba_pieter: {p.corr(mieszkania['liczba_pieter']):.2f}")
print(f"  średni błąd mediany ({p.median():g}):          {(p[zn] - p.median()).abs().mean():.2f}")
print(f"  średni błąd liczba_pieter // 2:           {(p[zn] - mieszkania.loc[zn, 'liczba_pieter'] // 2).abs().mean():.2f}")
mieszkania["pietro"] = p.fillna(mieszkania["liczba_pieter"] // 2).astype(int)
print("  uzupełnione piętra:", mieszkania.loc[mieszkania["pietro_brak"], ["id", "liczba_pieter", "pietro"]].to_dict("records"))

print()
for kol in ["parking", "balkon"]:
    zn_kol = mieszkania[kol].dropna()
    dom = zn_kol.mode()[0]
    print(f"{kol}: odsetek 'Tak' - tylko znane: {(zn_kol == 'Tak').mean():.1%} | "
          f"po uzupełnieniu dominantą ('{dom}'): {(mieszkania[kol].fillna(dom) == 'Tak').mean():.1%} | "
          f"z kategorią 'nieznane': {(mieszkania[kol].fillna('nieznane') == 'Tak').mean():.1%}")
    mieszkania[kol] = mieszkania[kol].fillna("nieznane")

print("\nbraki po uzupełnieniu:", int(mieszkania.isna().sum().sum()))
mieszkania.head()

### 🟢 Zadanie 13: porównaj przed i po

Wczytaj surowy plik jeszcze raz (z `na_values` z zadania 8) i porównaj z oczyszczoną wersją:
1. statystyki opisowe kolumn `powierzchnia`, `rok_budowy`, `cena_m2` (`describe()`),
2. histogram `rok_budowy` przed i po uzupełnieniu (dwa wykresy obok siebie, te same przedziały).

Czy uzupełnianie zmieniło rozkłady w istotny sposób?

In [ ]:
przed = pd.read_csv("nieruchomosci.csv", na_values=braki_slownik, keep_default_na=False)
kol = ["powierzchnia", "rok_budowy", "cena_m2"]

opis = pd.concat([
    przed[kol].describe().T.assign(wersja="przed (z brakami)"),
    mieszkania[kol].describe().T.assign(wersja="po (oczyszczone)"),
]).set_index("wersja", append=True).swaplevel().sort_index(level=1, sort_remaining=False)
display(opis.round(1))

bins = np.arange(1994, 2026, 2)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (tytul, dane) in zip(axes, [("przed uzupełnieniem", przed["rok_budowy"].dropna()),
                                     ("po uzupełnieniu (mediana wg stanu)", mieszkania["rok_budowy"])]):
    ax.hist(dane, bins=bins, edgecolor="white", color="steelblue")
    ax.set_title(f"Rok budowy — {tytul} (n = {len(dane)})")
    ax.set_xlabel("Rok budowy")
axes[0].set_ylabel("Liczba ofert")
plt.tight_layout()
plt.show()

*Twoja odpowiedź:* …


Powierzchnia i cena za m² praktycznie się nie zmieniły po uzupełnieniu brakujących danych. W przypadku roku budowy uzupełniono 7 brakujących wartości. Średnia wzrosła nieznacznie z 2010,3 do 2010,6, natomiast odchylenie standardowe spadło z 7,8 do 7,4. Na histogramie widać, że po uzupełnieniu zwiększyła się liczba ofert z rokiem budowy 2010-2012 oraz 2018-2020. Pojawiło się więc większe skupienie ofert w niektórych latach. Z tego powodu pozostawiono kolumnę rok_budowy_brak, która pozwala sprawdzić, w których ofertach rok budowy został uzupełniony.

### 🟢 Zadanie 14: zapisz decyzje

Uzupełnij tabelę. Dobra analiza danych to taka, którą ktoś inny może odtworzyć.

| Kolumna        | Jak zapisano braki / błędy       | Liczba braków | Decyzja                 | Uzasadnienie                                                                        |
| --- | --------- | ---- | --- | ------------ |
| `powierzchnia` | `"brak"` (2), `"N/A"` (4), `0` (3)           | 9 (20,0%)     | Odtworzyć jako `cena_total / cena_m2`           | Wartości można dokładnie obliczyć na podstawie pozostałych kolumn. Zależność potwierdzono w 34 z 34 kompletnych wierszy. Powierzchnia 0 m² jest niemożliwa. |
| `cena_m2`      | `-1` (1), `-999` (1)             | 2 (4,4%)      | Odtworzyć jako `cena_total / powierzchnia`      | Cenę można dokładnie obliczyć z pozostałych danych. Ujemne wartości nie mają sensu w tym przypadku.                                 |
| `rok_budowy`   | `"NA"` (4), `"brak"` (3)         | 7 (15,6%)     | Mediana według `stan` + flaga `rok_budowy_brak` | Stan najlepiej pomaga przewidzieć rok budowy. Średni błąd wynosi 3,0 roku, wobec 4,6 dla miasta i 6,5 dla mediany ogólnej.     |
| `pietro`       | `"brak"` (1), `-999` (2); `0` oznacza parter | 3 (6,7%)      | `liczba_pieter // 2` + flaga `pietro_brak`      | Piętro jest silnie związane z liczbą pięter w budynku (r = 0,88). Średni błąd tej metody wynosi 0,88, a mediany 1,62.          |
| `parking`      | `"?"`           | 5 (11,1%)     | Kategoria `"nieznane"` + flaga `parking_brak`   | Nie zakładamy, że brak informacji oznacza posiadanie parkingu.                   |
| `balkon`       | `"?"`           | 3 (6,7%)      | Kategoria `"nieznane"` + flaga `balkon_brak`    | Tak samo jak w przypadku parkingu, nie zgadujemy brakujących wartości.                            |
| `stan`         | `很好` (1 błędna wartość)     | 0 braków      | Poprawić na `"Bardzo dobry"`   | Jest to chińskie określenie oznaczające bardzo dobry . Poprawiono je, aby wszystkie wartości miały jednolity zapis.           |


### 🔵 Zadanie 15 (dla ambitnych): czy wybór metody ma znaczenie?

Uzupełnij `rok_budowy` na trzy sposoby: medianą całościową, medianą w grupach według miasta i medianą w grupach według stanu. Dla każdego wariantu policz korelację `rok_budowy` z `cena_m2`. Który wariant najmniej zmienia korelację w porównaniu z danymi bez braków?

In [ ]:
# TODO

*Twoja odpowiedź:* …